<a href="https://colab.research.google.com/github/miriam-silva/testesp2icoma/blob/main/prova_02_miriam_silva.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Prova 01:** IBM | Direção Diária:
**Aluna:** Miriam Silva Corrêa



# Pipeline 1 | Extração de Dados [Fase 1]

## Bibliotecas

In [ ]:
import requests
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

## Consumo da API (Alpha Vantage)

In [ ]:
API_KEY = 'XBWDB57V8PK8DW8W'
symbol = 'IBM'
url = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=IBM&apikey={API_KEY}&outputsize=compact'
response = requests.get(url)
data = response.json()

## Transformação para DataFrame

In [ ]:
if 'Time Series (Daily)' in data:
  df = pd.DataFrame.from_dict(data['Time Series (Daily)'], orient='index')
  df.columns = ['open', 'high', 'low', 'close', 'volume']
  df = df.astype(float)
  df.index = pd.to_datetime(df.index)
  df = df.sort_index()
  print("Dados carregados com sucesso!")
else:
  print("Erro na API:", data)

Dados carregados com sucesso!


In [ ]:
df.head()

,open,high,low,close,volume
2026-01-26,293.16,296.815,293.1400,296.33,3726890.0
2026-01-27,297.16,297.330,293.2700,293.86,2944722.0
2026-01-28,294.17,295.950,291.2601,294.16,5790347.0
2026-01-29,317.86,319.900,303.4700,309.24,10124929.0
2026-01-30,307.60,307.783,299.7300,306.70,5940669.0


# Pipeline 2 | Laborização (Feature Engineering) [Fase 2]

## Criação da Variável Alvo (Y)

In [ ]:
df['target'] = (df['close'] > df['open']).astype(int)

## Criação de Preditores (X) baseados em Lags

In [ ]:
df['variation'] = (df['close'] - df['open']) / df['open']
df['prev_variation'] = df['variation'].shift(1)
df['prev_variation_2'] = df['variation'].shift(2)
df['prev_variation_3'] = df['variation'].shift(3)
df['volume_log'] = np.log1p(df['volume'])

df['prev_volume'] = df['volume_log'].shift(1)
df['prev_volume_2'] = df['volume_log'].shift(2)

df_final = df[['prev_variation', 'prev_variation_2', 'prev_variation_3', 'prev_volume', 'prev_volume_2', 'target']].dropna()

print("Colunas extras removidas e valores nulos excluídos.")
df_final.head()

Colunas extras removidas e valores nulos excluídos.


,prev_variation,prev_variation_2,prev_variation_3,prev_volume,prev_volume_2,target
2026-01-29,-0.000034,-0.011105,0.010813,15.571703,14.895525,0
2026-01-30,-0.027119,-0.000034,-0.011105,16.130511,15.571703,0
2026-02-02,-0.002926,-0.027119,-0.000034,15.597332,16.130511,1
2026-02-03,0.023479,-0.002926,-0.027119,15.337469,15.597332,0
2026-02-04,-0.057907,0.023479,-0.002926,16.239961,15.337469,0


# Pipeline 3 | Tratamento e Normalização [Fase 3]

In [ ]:
df_final.head()

,prev_variation,prev_variation_2,prev_variation_3,prev_volume,prev_volume_2,target
2026-01-29,-0.000034,-0.011105,0.010813,15.571703,14.895525,0
2026-01-30,-0.027119,-0.000034,-0.011105,16.130511,15.571703,0
2026-02-02,-0.002926,-0.027119,-0.000034,15.597332,16.130511,1
2026-02-03,0.023479,-0.002926,-0.027119,15.337469,15.597332,0
2026-02-04,-0.057907,0.023479,-0.002926,16.239961,15.337469,0


## Verificação de valores nulos

In [ ]:
df_final.isnull().sum()

,0
prev_variation,0
prev_variation_2,0
prev_variation_3,0
prev_volume,0
prev_volume_2,0
target,0


## Estatísticas descritivas

In [ ]:
df_final.describe()

,prev_variation,prev_variation_2,prev_variation_3,prev_volume,prev_volume_2,target
count,97.000000,97.000000,97.000000,97.000000,97.000000,97.000000
mean,-0.001980,-0.002092,-0.001856,15.697479,15.692192,0.474227
std,0.025717,0.025733,0.025745,0.499245,0.505020,0.501929
min,-0.121948,-0.121948,-0.121948,14.672507,14.672507,0.000000
25%,-0.012082,-0.012082,-0.011105,15.387766,15.378166,0.000000
50%,-0.001019,-0.002313,-0.001019,15.585256,15.585256,0.000000
75%,0.009595,0.009595,0.010526,15.939114,15.939114,1.000000
max,0.087669,0.087669,0.087669,17.308381,17.308381,1.000000


## Remoção de Outliers

In [ ]:
df_final = df_final[
    (np.abs(df_final['prev_variation']) < 3 * df_final['prev_variation'].std()) &
    (np.abs(df_final['prev_variation_2']) < 3 * df_final['prev_variation_2'].std()) &
    (np.abs(df_final['prev_variation_3']) < 3 * df_final['prev_variation_3'].std())
]

## Seleção de Variáveis (X e y)

In [ ]:
X = df_final[['prev_variation', 'prev_variation_2', 'prev_variation_3', 'prev_volume', 'prev_volume_2']]
y = df_final['target']

## Análise de Balanceamento das Classes

In [ ]:
print(y.value_counts())

target
0    49
1    42
Name: count, dtype: int64


## Divisão Treino/Teste

In [ ]:
tamanho = int(len(X)*0.9)

X_train = X[:tamanho]
X_test = X[tamanho:]

y_train = y[:tamanho]
y_test = y[tamanho:]

## Normalização (StandardScaler)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Pipeline 4 | Decision Tree [Fase 4]

In [ ]:
arvore_ibm = DecisionTreeClassifier(criterion='entropy', random_state=0)
arvore_ibm.fit(X_train, y_train)

DecisionTreeClassifier(criterion='entropy', random_state=0)

In [ ]:
previsoes_arvore = arvore_ibm.predict(X_test)

In [ ]:
previsoes_arvore

array([1, 0, 0, 0, 1, 0, 0, 1, 1, 0])

In [ ]:
accuracy_score(y_test, previsoes_arvore)

0.5

In [ ]:
confusion_matrix(y_test, previsoes_arvore)

array([[5, 4],
       [1, 0]])

In [ ]:
print(classification_report(y_test, previsoes_arvore))

              precision    recall  f1-score   support

           0       0.83      0.56      0.67         9
           1       0.00      0.00      0.00         1

    accuracy                           0.50        10
   macro avg       0.42      0.28      0.33        10
weighted avg       0.75      0.50      0.60        10



## Avaliação do modelo

In [ ]:
print(classification_report(y_test, y_pred_arvore))

              precision    recall  f1-score   support

           0       1.00      0.22      0.36         9
           1       0.12      1.00      0.22         1

    accuracy                           0.30        10
   macro avg       0.56      0.61      0.29        10
weighted avg       0.91      0.30      0.35        10



# Pipeline 5 | Modelagem por Random Forest [Fase 5]

In [ ]:
random_ibm = RandomForestClassifier(n_estimators=10, criterion='entropy', random_state=0)
random_ibm.fit(X_train, y_train)
random_ibm

RandomForestClassifier(criterion='entropy', n_estimators=10, random_state=0)

In [ ]:
previsoes_random = random_ibm.predict(X_test)

In [ ]:
previsoes_random

array([1, 0, 1, 0, 1, 1, 0, 1, 0, 1])

In [ ]:
accuracy_score(y_test, previsoes_random)

0.5

In [ ]:
confusion_matrix(y_test, previsoes_random)

array([[4, 5],
       [0, 1]])

In [ ]:
print(classification_report(y_test, previsoes_random))

              precision    recall  f1-score   support

           0       1.00      0.44      0.62         9
           1       0.17      1.00      0.29         1

    accuracy                           0.50        10
   macro avg       0.58      0.72      0.45        10
weighted avg       0.92      0.50      0.58        10



# Pipeline 6 | SVM [Fase 6]

In [ ]:
svm_ibm = SVC(kernel='rbf', C=2.0, random_state=1)
svm_ibm.fit(X_train, y_train)
svm_ibm

SVC(C=2.0, random_state=1)

In [ ]:
svm_previsoes = svm_ibm.predict(X_test)
svm_previsoes

array([0, 1, 1, 1, 1, 1, 1, 1, 0, 1])

In [ ]:
accuracy_score(y_test, svm_previsoes)

0.3

In [ ]:
confusion_matrix(y_test, svm_previsoes)

array([[2, 7],
       [0, 1]])

In [ ]:
print(classification_report(y_test, svm_previsoes))

              precision    recall  f1-score   support

           0       1.00      0.22      0.36         9
           1       0.12      1.00      0.22         1

    accuracy                           0.30        10
   macro avg       0.56      0.61      0.29        10
weighted avg       0.91      0.30      0.35        10



## Armazenamento de métricas

In [ ]:
acc_svm = accuracy_score(y_test, y_pred_svm)
prec_svm = precision_score(y_test, y_pred_svm)
rec_svm = recall_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm)

# Pipeline 7 | Redes Neurais [Fase 7 - IBM Base]

In [ ]:
redeneural_ibm = MLPClassifier(verbose=True, max_iter=1000, tol=0.000010, solver='adam', hidden_layer_sizes=(100), activation='relu')
redeneural_ibm.fit(X_train, y_train)
redeneural_ibm

Iteration 1, loss = 0.68084209
Iteration 2, loss = 0.67783545
Iteration 3, loss = 0.67497082
Iteration 4, loss = 0.67222120
Iteration 5, loss = 0.66956566
Iteration 6, loss = 0.66702342
Iteration 7, loss = 0.66458806
Iteration 8, loss = 0.66223747
Iteration 9, loss = 0.65998676
Iteration 10, loss = 0.65781204
Iteration 11, loss = 0.65569213
Iteration 12, loss = 0.65362819
Iteration 13, loss = 0.65162656
Iteration 14, loss = 0.64967066
Iteration 15, loss = 0.64777506
Iteration 16, loss = 0.64594817
Iteration 17, loss = 0.64415696
Iteration 18, loss = 0.64240527
Iteration 19, loss = 0.64068544
Iteration 20, loss = 0.63898553
Iteration 21, loss = 0.63730407
Iteration 22, loss = 0.63563171
Iteration 23, loss = 0.63400299
Iteration 24, loss = 0.63240361
Iteration 25, loss = 0.63081955
Iteration 26, loss = 0.62926992
Iteration 27, loss = 0.62773315
Iteration 28, loss = 0.62621165
Iteration 29, loss = 0.62469931
Iteration 30, loss = 0.62320829
Iteration 31, loss = 0.62172109
Iteration 32, los

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


MLPClassifier(hidden_layer_sizes=100, max_iter=1000, tol=1e-05, verbose=True)

In [ ]:
redeneural_ibm.out_activation_

'logistic'

In [ ]:
previsoes_redeneural = redeneural_ibm.predict(X_test)
previsoes_redeneural

array([0, 0, 1, 1, 1, 1, 0, 0, 0, 1])

In [ ]:
accuracy_score(y_test, previsoes_redeneural)

0.6

In [ ]:
confusion_matrix(y_test, previsoes_redeneural)

array([[5, 4],
       [0, 1]])

In [ ]:
print(classification_report(y_test, previsoes_redeneural))

              precision    recall  f1-score   support

           0       1.00      0.56      0.71         9
           1       0.20      1.00      0.33         1

    accuracy                           0.60        10
   macro avg       0.60      0.78      0.52        10
weighted avg       0.92      0.60      0.68        10



# Pipeline 8 | KNN [Fase 8 - IBM Base]

In [ ]:
parametros = {'n_neighbors': [3, 5, 10, 20], 'p': [1, 2]}

In [ ]:
grid_ibm = GridSearchCV(estimator=KNeighborsClassifier(), param_grid=parametros)
grid_ibm.fit(X_train, y_train)

GridSearchCV(estimator=KNeighborsClassifier(),
             param_grid={'n_neighbors': [3, 5, 10, 20], 'p': [1, 2]})

In [ ]:
melhor_parametros_ibm = grid_ibm.best_params_
melhor_resultado_ibm = grid_ibm.best_score_

In [ ]:
print(melhor_parametros_ibm)
print(melhor_resultado_ibm)

{'n_neighbors': 20, 'p': 1}
0.5426470588235295


In [ ]:
knn_ibm = KNeighborsClassifier(n_neighbors=melhor_parametros_ibm['n_neighbors'], p=melhor_parametros_ibm['p'])
knn_ibm.fit(X_train, y_train)
knn_ibm

KNeighborsClassifier(n_neighbors=20, p=1)

In [ ]:
previsoes_knn = knn_ibm.predict(X_test)

In [ ]:
previsoes_knn

array([1, 0, 1, 1, 1, 1, 1, 0, 0, 1])

In [ ]:
accuracy_score(y_test, previsoes_knn)

0.4

In [ ]:
confusion_matrix(y_test, previsoes_knn)

array([[3, 6],
       [0, 1]])

In [ ]:
confusion_matrix(y_test, previsoes_knn)

array([[3, 6],
       [0, 1]])

In [ ]:
print(classification_report(y_test, previsoes_knn))

              precision    recall  f1-score   support

           0       1.00      0.33      0.50         9
           1       0.14      1.00      0.25         1

    accuracy                           0.40        10
   macro avg       0.57      0.67      0.38        10
weighted avg       0.91      0.40      0.47        10



# Pipeline 9 | Comparação dos modelos [Fase 9]

In [ ]:
resultado_modelos = pd.DataFrame({
    'Modelo': ['Decision Tree', 'Random Forest', 'SVM', 'Redes Neurais', 'KNN'],
    'Acurácia': [
        accuracy_score(y_test, previsoes_arvore),
        accuracy_score(y_test, previsoes_random),
        accuracy_score(y_test, svm_previsoes),
        accuracy_score(y_test, previsoes_redeneural),
        accuracy_score(y_test, previsoes_knn)
    ]
})

resultado_modelos.sort_values(by='Acurácia', ascending=False)

,Modelo,Acurácia
3,Redes Neurais,0.6
0,Decision Tree,0.5
1,Random Forest,0.5
4,KNN,0.4
2,SVM,0.3
